# Modify and write runoff maps (r05)
This notebook merges runoff mapping files using a 250 km radius in the northern hemisphere and a 100 km radius in the southern hemisphere.

In [1]:
# Import libraries
import numpy as np
import os
import xarray as xr
from netCDF4 import Dataset
from scipy import sparse
import time

In [2]:
def load_runoff_map(nc_path):
    # Load NetCDF data (subtract 1 for 0-based indexing)
    with Dataset(nc_path, 'r') as nc:
        row = nc.variables['row'][:] - 1
        col = nc.variables['col'][:] - 1
        s_vals = nc.variables['S'][:]

    # Matrix dimensions (540 * 480 = 259200)
    N = 540 * 480
    # Create CSR sparse matrix
    S = sparse.csr_matrix((s_vals, (row, col)), shape=(N, N))
    return S

In [3]:
def combine_matrices_efficiently(S_sh, S_nh, yc_a):
    # Define the mask (True for Southern Hemisphere)
    mask = yc_a < 0.0
    num_cols = S_nh.shape[1]
    
    # 2. Determine the structure (indptr) of the new matrix
    # indptr[i+1] - indptr[i] is the number of non-zero entries in column i
    counts_nh = np.diff(S_nh.indptr)
    counts_sh = np.diff(S_sh.indptr)
    new_counts = np.where(mask, counts_sh, counts_nh)
    
    new_indptr = np.empty(num_cols + 1, dtype=S_nh.indptr.dtype)
    new_indptr[0] = 0
    np.cumsum(new_counts, out=new_indptr[1:])
    
    # 3. Pre-allocate the data and indices arrays
    # This is the most memory-efficient way: one allocation for exactly the size needed
    nnz = new_indptr[-1]
    new_data = np.empty(nnz, dtype=S_nh.dtype)
    new_indices = np.empty(nnz, dtype=S_nh.indices.dtype)
    
    # 4. Copy data in contiguous blocks to minimize Python loop overhead
    # We find where the mask changes (e.g., from NH to SH)
    changes = np.where(mask[1:] != mask[:-1])[0] + 1
    boundaries = np.concatenate(([0], changes, [num_cols]))
    
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        # Determine which source matrix to use for this block of columns
        src = S_sh if mask[start] else S_nh
        
        # Identify the range of indices in the source's data/indices arrays
        src_start_idx, src_end_idx = src.indptr[start], src.indptr[end]
        # Identify where to put them in the new arrays
        tgt_start_idx, tgt_end_idx = new_indptr[start], new_indptr[end]
        
        new_data[tgt_start_idx:tgt_end_idx] = src.data[src_start_idx:src_end_idx]
        new_indices[tgt_start_idx:tgt_end_idx] = src.indices[src_start_idx:src_end_idx]
        
    # 5. Build the final sparse matrix
    return sparse.csc_matrix((new_data, new_indices, new_indptr), shape=S_nh.shape)

In [4]:
def merge_runoff_maps(nc_path_sh, nc_path_nh):
    # Load runoff maps
    print('loading runoff masks')
    S_nh = load_runoff_map(nc_path_nh)
    S_sh = load_runoff_map(nc_path_sh)

    # Use CSC format for efficient column modification
    print('converting to CSC')
    S_nh = S_nh.tocsc()
    S_sh = S_sh.tocsc()

    # Get latitudes in source grid
    with Dataset(nc_path_nh, 'r') as nc:
        yc_a = nc.variables['yc_a'][:]

    # Create new mask
    print('Making new mask')
    S_new = combine_matrices_efficiently(S_sh, S_nh, yc_a)

    # Get non-zero elements for saving (Add 1 back to match Fortran 1-based expectations)
    r_final, c_final = S_new.nonzero()
    s_final = np.array(S_new[r_final, c_final]).flatten()

    return (r_final+1, c_final+1, s_final)

In [5]:
def write_merged_map(nc_path_sh, nc_path_nh, new_nc_path):
    print('Modifying the map:')
    r_modified, c_modified, s_modified = merge_runoff_maps(nc_path_sh, nc_path_nh)
    print('Writing the modified map:')
    ds_runoff_map = xr.open_dataset(nc_path_sh)
    with Dataset(new_nc_path, "w", format="NETCDF4") as rootgrp:
        rootgrp.title = "runoff map: r05 -> tx2_3v3, nearest neighbor and smoothed, merging " + nc_path_sh + " and " + nc_path_nh
        rootgrp.author = 'Ian Grooms (ian.grooms@colorado.edu)'
        rootgrp.history= "File created " + time.ctime(time.time()) + " using Merge_runoff_maps.ipynb"
        rootgrp.conventions = "NCAR-CCSM"
        rootgrp.domain_a = "/glade/p/cesm/cseg/inputdata/lnd/clm2/rtmdata/rdirc.05.061026"
        rootgrp.domain_b = "/glade/work/gmarques/cesm/tx2_3/mesh/tx2_3v3_260305_SCRIP.nc"
        rootgrp.createDimension('n_a', 259200)
        rootgrp.createDimension('n_b', 259200)
        rootgrp.createDimension('ni_a', 720)
        rootgrp.createDimension('ni_b', 540)
        rootgrp.createDimension('nj_a', 360)
        rootgrp.createDimension('nj_b', 480)
        rootgrp.createDimension('nv_a', 4)
        rootgrp.createDimension('nv_b', 4)
        rootgrp.createDimension('src_grid_rank', 2)
        rootgrp.createDimension('dst_grid_rank', 2)
        rootgrp.createDimension('n_s', s_modified.shape[0])
        rootgrp.createVariable('xc_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["xc_a"].units = "degrees east"
        rootgrp["xc_a"].long_name = "longitude of grid cell center (input)"
        rootgrp["xc_a"][:] = ds_runoff_map.xc_a.data[:]
        rootgrp.createVariable('yc_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["yc_a"].units = "degrees north"
        rootgrp["yc_a"].long_name = "latitude of grid cell center (input)"
        rootgrp["yc_a"][:] = ds_runoff_map.yc_a.data[:]
        rootgrp.createVariable('xv_a','f8',('n_a','nv_a'),fill_value=np.nan)
        rootgrp["xv_a"].units = "degrees east"
        rootgrp["xv_a"].long_name = "longitude of grid cell vertices (input)"
        rootgrp["xv_a"][:] = ds_runoff_map.xv_a.data[:,:]
        rootgrp.createVariable('yv_a','f8',('n_a','nv_a'),fill_value=np.nan)
        rootgrp["yv_a"].units = "degrees north"
        rootgrp["yv_a"].long_name = "latitude of grid cell vertices (input)"
        rootgrp["yv_a"][:] = ds_runoff_map.yv_a.data[:,:]
        rootgrp.createVariable('mask_a','i4',('n_a',))
        rootgrp["mask_a"].long_name = "domain mask (input)"
        rootgrp["mask_a"][:] = ds_runoff_map.mask_a.data[:]
        rootgrp.createVariable('area_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["area_a"].long_name = "area of cell (input)"
        rootgrp["area_a"][:] = ds_runoff_map.area_a.data[:]
        rootgrp.createVariable('frac_a','f8',('n_a',),fill_value=np.nan)
        rootgrp["frac_a"].long_name = "fraction of domain intersection (input)"
        rootgrp["frac_a"][:] = ds_runoff_map.frac_a.data[:]
        rootgrp.createVariable('src_grid_dims','i4',('src_grid_rank',))
        rootgrp["src_grid_dims"][:] = ds_runoff_map.src_grid_dims.data[:]

        rootgrp.createVariable('xc_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["xc_b"].units = "degrees east"
        rootgrp["xc_b"].long_name = "longitude of grid cell center (output)"
        rootgrp["xc_b"][:] = ds_runoff_map.xc_b.data[:]
        rootgrp.createVariable('yc_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["yc_b"].units = "degrees north"
        rootgrp["yc_b"].long_name = "latitude of grid cell center (output)"
        rootgrp["yc_b"][:] = ds_runoff_map.yc_b.data[:]
        rootgrp.createVariable('xv_b','f8',('n_b','nv_b'),fill_value=np.nan)
        rootgrp["xv_b"].units = "degrees east"
        rootgrp["xv_b"].long_name = "longitude of grid cell vertices (output)"
        rootgrp["xv_b"][:] = ds_runoff_map.xv_b.data[:,:]
        rootgrp.createVariable('yv_b','f8',('n_b','nv_b'),fill_value=np.nan)
        rootgrp["yv_b"].units = "degrees north"
        rootgrp["yv_b"].long_name = "latitude of grid cell vertices (output)"
        rootgrp["yv_b"][:] = ds_runoff_map.yv_b.data[:,:]
        rootgrp.createVariable('mask_b','i4',('n_b',))
        rootgrp["mask_b"].long_name = "domain mask (output)"
        rootgrp["mask_b"][:] = ds_runoff_map.mask_b.data[:]
        rootgrp.createVariable('area_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["area_b"].long_name = "area of cell (output)"
        rootgrp["area_b"][:] = ds_runoff_map.area_b.data[:]
        rootgrp.createVariable('frac_b','f8',('n_b',),fill_value=np.nan)
        rootgrp["frac_b"].long_name = "fraction of domain intersection (output)"
        rootgrp["frac_b"][:] = ds_runoff_map.frac_b.data[:]
        rootgrp.createVariable('dst_grid_dims','i4',('dst_grid_rank',))
        rootgrp["dst_grid_dims"][:] = ds_runoff_map.dst_grid_dims.data[:]

        rootgrp.createVariable('S','f8',('n_s',), fill_value=np.nan)
        rootgrp["S"].long_name = "sparse matrix for mapping S:a->b"
        rootgrp["S"][:] = s_modified
        rootgrp.createVariable('row','i4',('n_s',))
        rootgrp["row"].long_name = "row corresponding to matrix elements"
        rootgrp.createVariable('col','i4',('n_s',))
        rootgrp["row"][:] = r_modified
        rootgrp["col"].long_name = "column corresponding to matrix elements"
        rootgrp["col"][:] = c_modified
    print('Done')

In [6]:
# Make and write merged map:
nc_path_sh = '/glade/work/gmarques/cesm/tx2_3/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e100r100_260306.nc'
nc_path_nh = '/glade/work/gmarques/cesm/tx2_3/runoff_mapping/map_r05_to_tx2_3v3_nnsm_e250r250_260306.nc'
new_nc_path = 'map_r05_to_tx2_3v3_nnsm_e100r100sh_e250r250nh_merged_260307.nc'
write_merged_map(nc_path_sh, nc_path_nh, new_nc_path)

Modifying the map:
loading runoff masks
converting to CSC
Making new mask
Writing the modified map:
Done


The merged map should satisfy `S.T@*area_b = area_a`.